# Notebook 02
## Understanding How an SLM Thinks

In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Notebook Ready")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Notebook Ready


# 1. The First Layer - Token Embeddings

In [4]:
embedding_layer = model.model.embed_tokens

print(embedding_layer)

Embedding(151936, 896)


In [5]:
embedding_matrix = embedding_layer.weight

print(embedding_matrix.shape)

torch.Size([151936, 896])


# 2. Looking Inside a Word

In [6]:
word = "law"

tokens = tokenizer(word)

print(tokens)

{'input_ids': [19915], 'attention_mask': [1]}


In [7]:
token_id = tokens["input_ids"][0]

print(token_id)

19915


In [8]:
embedding = embedding_matrix[token_id]

print(embedding.shape)

torch.Size([896])


In [9]:
print(embedding[:25])

tensor([-0.0040, -0.0056, -0.0001, -0.0310,  0.0035,  0.0171, -0.0030,  0.0222,
         0.0066, -0.0002,  0.0184,  0.0085,  0.0281,  0.0170,  0.0038, -0.0023,
         0.0280,  0.0064,  0.0076,  0.0248, -0.0019,  0.0134,  0.0063, -0.0082,
        -0.0096], device='cuda:0', dtype=torch.float16,
       grad_fn=<SliceBackward0>)


# 3. Following Information Through the Network

In [10]:
prompt = "The capital of India is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

In [11]:
with torch.no_grad():

    outputs = model(
        **inputs,
        output_hidden_states=True
    )

In [12]:
hidden_states = outputs.hidden_states

print(len(hidden_states))

25


# 4. Why Are There 25 Hidden States?

In [13]:
print(type(hidden_states))

print()

print(f"Number of Hidden States : {len(hidden_states)}")

<class 'tuple'>

Number of Hidden States : 25


In [14]:
for i, state in enumerate(hidden_states):

    print(
        f"Hidden State {i:02d}",
        state.shape
    )

Hidden State 00 torch.Size([1, 5, 896])
Hidden State 01 torch.Size([1, 5, 896])
Hidden State 02 torch.Size([1, 5, 896])
Hidden State 03 torch.Size([1, 5, 896])
Hidden State 04 torch.Size([1, 5, 896])
Hidden State 05 torch.Size([1, 5, 896])
Hidden State 06 torch.Size([1, 5, 896])
Hidden State 07 torch.Size([1, 5, 896])
Hidden State 08 torch.Size([1, 5, 896])
Hidden State 09 torch.Size([1, 5, 896])
Hidden State 10 torch.Size([1, 5, 896])
Hidden State 11 torch.Size([1, 5, 896])
Hidden State 12 torch.Size([1, 5, 896])
Hidden State 13 torch.Size([1, 5, 896])
Hidden State 14 torch.Size([1, 5, 896])
Hidden State 15 torch.Size([1, 5, 896])
Hidden State 16 torch.Size([1, 5, 896])
Hidden State 17 torch.Size([1, 5, 896])
Hidden State 18 torch.Size([1, 5, 896])
Hidden State 19 torch.Size([1, 5, 896])
Hidden State 20 torch.Size([1, 5, 896])
Hidden State 21 torch.Size([1, 5, 896])
Hidden State 22 torch.Size([1, 5, 896])
Hidden State 23 torch.Size([1, 5, 896])
Hidden State 24 torch.Size([1, 5, 896])


In [15]:
state = hidden_states[0]

print(f"Batch Size : {state.shape[0]}")

print(f"Sequence Length : {state.shape[1]}")

print(f"Embedding Size : {state.shape[2]}")

Batch Size : 1
Sequence Length : 5
Embedding Size : 896


In [16]:
print("Embedding Output")

print(hidden_states[0].shape)

print()

print("After Layer 1")

print(hidden_states[1].shape)

print()

print("After Final Layer")

print(hidden_states[-1].shape)

Embedding Output
torch.Size([1, 5, 896])

After Layer 1
torch.Size([1, 5, 896])

After Final Layer
torch.Size([1, 5, 896])


# 5. Watching The Sentence Evolve

In [17]:
token_index = 0

print("Original Embedding")

print(hidden_states[0][0, token_index, :15])

print()

print("After Layer 1")

print(hidden_states[1][0, token_index, :15])

print()

print("After Final Layer")

print(hidden_states[-1][0, token_index, :15])

Original Embedding
tensor([-0.0320, -0.0017,  0.0121, -0.0156,  0.0156, -0.0295,  0.0024, -0.0156,
        -0.0109,  0.0073,  0.0009, -0.0043, -0.0085,  0.0123,  0.0068],
       device='cuda:0', dtype=torch.float16)

After Layer 1
tensor([-0.3179, -0.2010, -0.0605, -0.1769,  0.0230, -0.0173,  0.0815,  0.0048,
         0.0178,  0.0618, -0.0262,  0.1199,  0.0863, -0.0145, -0.0488],
       device='cuda:0', dtype=torch.float16)

After Final Layer
tensor([-4.4336, -6.6055,  6.5273,  0.7925, -4.3320,  1.5469,  4.0586, -3.8379,
        -4.8125,  1.0527, -9.7031,  6.4648, -0.9941, -3.4629,  4.9336],
       device='cuda:0', dtype=torch.float16)


# 6. Next Token Prediction

In [18]:
logits = outputs.logits

print(logits.shape)

torch.Size([1, 5, 151936])


In [19]:
last_token_logits = logits[:, -1, :]

print(last_token_logits.shape)

torch.Size([1, 151936])


# 7. Converting Scores Into Probabilities

In [20]:
probabilities = torch.softmax(
    last_token_logits,
    dim=-1
)

print(probabilities.shape)

torch.Size([1, 151936])


In [21]:
print(probabilities.sum())

tensor(1., device='cuda:0', dtype=torch.float16)


# 8. What Was The Model Thinking?

In [22]:
top_probs, top_indices = torch.topk(
    probabilities,
    k=10
)

In [23]:
print("=" * 60)
print("Top 10 Predictions")
print("=" * 60)

for probability, token in zip(
    top_probs[0],
    top_indices[0]
):

    word = tokenizer.decode([token])

    print(
        f"{word:<20} {probability.item():.5f}"
    )

Top 10 Predictions
 located             0.20020
 New                 0.08478
 known               0.07367
 the                 0.06107
 __                  0.06012
:
                   0.04681
 ______              0.04297
 ____                0.03192

                    0.02373
:

                  0.02354


# 9. Chat Templates

In [24]:
messages = [
    {
        "role": "user",
        "content": "What is the capital of India?"
    }
]

In [25]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of India?<|im_end|>
<|im_start|>assistant



In [26]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

In [27]:
with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=100
    )

In [28]:
answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(answer)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is the capital of India?
assistant
The capital of India is New Delhi. It is located in the northern part of the country and serves as the administrative and cultural center for the Indian subcontinent. The city has been the capital since 1957 when it was established as the seat of government under the British Raj. It is known for its rich history, vibrant culture, and diverse population.


# 10. Controlling Model Generation

In [29]:
question = "Write one sentence about Artificial Intelligence."

messages = [
    {
        "role":"user",
        "content":question
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

In [30]:
output = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.0,
    do_sample=False
)

print(tokenizer.decode(
    output[0],
    skip_special_tokens=True
))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write one sentence about Artificial Intelligence.
assistant
Artificial intelligence is a subset of computer science that focuses on the development and application of intelligent machines capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and decision


In [31]:
output = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=1.0,
    do_sample=True
)

print(tokenizer.decode(
    output[0],
    skip_special_tokens=True
))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write one sentence about Artificial Intelligence.
assistant
Artificial intelligence has the ability to process and analyze large amounts of data quickly and accurately, enabling machines to learn from experience and make decisions that can improve over time.


In [32]:
output = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=2.0,
    do_sample=True
)

print(tokenizer.decode(
    output[0],
    skip_special_tokens=True
))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write one sentence about Artificial Intelligence.
assistant
Artificial Intelligence, or AI, encompasses a broad category that involves the simulation of human intelligence processes in machine software to perform tasks that would typically require human cognition, learning, reasoning, and perception.


# 11. Complete SLM Pipeline

In [33]:
prompt = "What is Machine Learning?"

messages = [
    {
        "role":"user",
        "content":prompt
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=100
    )

answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(answer)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is Machine Learning?
assistant
Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data without being explicitly programmed. In other words, it's about creating systems that can improve their performance over time through experience and feedback.

Key characteristics of machine learning include:

1. **Learning**: The ability to identify patterns in data and use those patterns to make predictions.
2. **Generalization**: The ability to perform well on new data that has


What Just Happened

User Prompt
      │
      ▼
Chat Template
      │
      ▼
Tokenizer
      │
      ▼
Input IDs
      │
      ▼
Embeddings
      │
      ▼
24 Transformer Layers
      │
      ▼
Hidden States
      │
      ▼
Logits
      │
      ▼
Softmax
      │
      ▼
Next Token
      │
      ▼
Repeat Until EOS
      │
      ▼
Tokenizer Decode
      │
      ▼
Final Response

# Notebook Summary

Today we built and understood our first Small Language Model.

We learned:

- Loading pretrained models
- Tokenization
- Vocabulary
- Embeddings
- Hidden States
- Transformer Layers
- Logits
- Softmax
- Next Token Prediction
- Chat Templates
- Text Generation

We now understand the complete inference pipeline of a pretrained SLM.

Next notebook:

Building our first Retrieval-Augmented Generation (RAG) system.